# Lesson 4: Two students, one file

Maya is fixing the dosage table (#21). Lee is adding units to it (#22). Both change `chapters/dosage.md`. Neither did anything wrong, but whoever lands second will have to bring the other's change in first. Symphony cannot stop the overlap, so it does the next best thing: the board lists every file two pieces of work share, and says in what order they should land.

## Step 0: open a practice folder

Run this cell first. It makes an empty practice folder, `$LAB`, and fetches the claim-protocol library; every later block starts with `cd "$LAB"`, so nothing touches your real projects.

In [ ]:
# The practice folder, and the newest release of the public claim-protocol library.
import os, re, subprocess, tempfile
tags = subprocess.run(["git", "ls-remote", "--tags", "https://github.com/OpenSpineConsortium/symphony-protocol"], capture_output=True, text=True, check=True).stdout
releases = sorted({tuple(map(int, m)) for m in re.findall(r"refs/tags/v(\d+)\.(\d+)\.(\d+)$", tags, re.M)})
TAG = "v%d.%d.%d" % releases[-1]
LAB = tempfile.mkdtemp(prefix="symphony-lesson-")
LIBRARY = os.path.join(tempfile.mkdtemp(), "symphony-protocol")
subprocess.run(["git", "-c", "advice.detachedHead=false", "clone", "--quiet", "--depth", "1",
                "--branch", TAG, "https://github.com/OpenSpineConsortium/symphony-protocol", LIBRARY], check=True)
os.environ.update({"GIT_CONFIG_GLOBAL": "/dev/null", "GIT_CONFIG_NOSYSTEM": "1", "GIT_AUTHOR_DATE": "2026-10-20T09:00:00Z", "GIT_COMMITTER_DATE": "2026-10-20T09:00:00Z", "GIT_CONFIG_COUNT": "4", "GIT_CONFIG_KEY_0": "gc.auto", "GIT_CONFIG_VALUE_0": "0", "GIT_CONFIG_KEY_1": "gc.autoDetach", "GIT_CONFIG_VALUE_1": "false", "GIT_CONFIG_KEY_2": "maintenance.auto", "GIT_CONFIG_VALUE_2": "false", "GIT_CONFIG_KEY_3": "maintenance.autoDetach", "GIT_CONFIG_VALUE_3": "false"})
os.environ.update(LAB=LAB, PYTHONPATH=LIBRARY)
print("symphony-protocol", TAG, "; practice folder:", LAB)

## Step 1: describe two pull requests that share a file

Maya opened her draft pull request first. Lee opened his an hour later, but has already marked it ready for review, which deleted his claim file, so the board names him by his `worker:lee` label. Besides the open issues, the board reads each claimed issue's labels (`claim_issues`), which is where it learns that a claim's issue is a blocker.

In [ ]:
%%bash
cd "$LAB"
cat > course.json <<'JSON'
{
  "sha": "0123abcdef0123abcdef0123abcdef0123abcdef",
  "default_branch": "main",
  "issues": [
    {"number": 21, "title": "Fix the dosage table", "labels": [{"name": "claimed"}, {"name": "worker:maya"}]},
    {"number": 22, "title": "Add units to the dosage table", "labels": [{"name": "claimed"}, {"name": "worker:lee"}]}
  ],
  "pulls": [
    {"number": 50, "title": "Fix the dosage table", "head": {"ref": "claude/21", "sha": "aaaa"}, "draft": true,
     "created_at": "2026-10-20T09:00:00Z", "mergeable_state": "clean", "updated_at": "2026-10-20T09:00:00Z",
     "files": ["chapters/dosage.md"]},
    {"number": 51, "title": "Add units", "head": {"ref": "claude/22", "sha": "bbbb"}, "draft": false,
     "created_at": "2026-10-20T10:00:00Z", "mergeable_state": "clean", "updated_at": "2026-10-20T10:00:00Z",
     "files": ["chapters/dosage.md", "chapters/units.md"]}
  ],
  "branches": ["main", "claude/21", "claude/22"],
  "claims": {"21": "- Worker: maya\n", "22": null},
  "claim_issues": {
    "21": {"state": "open", "labels": ["claimed", "worker:maya"]},
    "22": {"state": "open", "labels": ["claimed", "worker:lee"]}
  },
  "merged": []
}
JSON
echo "written"

You should see lines like these:

```
written
```

## Step 2: read the Overlaps section

In [ ]:
%%bash
cd "$LAB"
python3 -m sdlc_protocol board --dry-run course.json > STATE.md
grep -A5 "^## Overlaps" STATE.md

You should see lines like these:

```
## Overlaps
land the work in this order: a blocker first, then work marked ready, then the oldest
| chapters/dosage.md | #51, #50 |
```

Lee's #51 lands first, even though Maya started earlier, because it is marked ready: it waits only for its merge, while Maya's draft is still changing. `chapters/units.md` is not listed, because only Lee touches it.

## Step 3: what if Maya's work were a blocker?

Say the dosage error makes the PDF build fail for everyone, so the teacher labels #21 a blocker. A blocker lands first, whatever else is ready.

In [ ]:
%%bash
cd "$LAB"
python3 - <<'PY'
import json
course = json.load(open("course.json"))
course["issues"][0]["labels"].append({"name": "blocker"})
course["claim_issues"]["21"]["labels"].append("blocker")
json.dump(course, open("course.json", "w"))
PY
python3 -m sdlc_protocol board --dry-run course.json > STATE.md
grep "chapters/dosage.md |" STATE.md

You should see lines like these:

```
| chapters/dosage.md | #50, #51 |
```

## What happens next in a real repository

The coordinator, a step of the board's own workflow, comments once on each piece of work with the shared file and the order, so Maya and Lee hear about it without anyone watching the board. When #51 merges, the board tells Maya's pull request that `main` moved, and she brings it in ([PROTOCOL.md](../../agents/PROTOCOL.md), Overlaps and Work in the open).

## What you learned

- Overlaps are allowed; surprises are not. The board names every shared file and the landing order: a blocker first, then work marked ready, then the oldest.
- You have now used claims, the push guard, the board's sections and Overlaps: the parts of Symphony every worker meets, person or Claude. [capabilities.md](../../start/capabilities.md) shows what else a project can do.